# 第1章：机器学习基础

> **李宏毅老师经典开篇：** "机器学习就是让机器具备**找一个函数**的能力。"

## 本章知识导图

```
机器学习基础 (从线性模型到深度学习的第一步)
│
├── 1.1 案例学习：预测YouTube频道观看人数
│   ├── 问题定义：输入=历史信息 → 输出=明日观看次数
│   ├── Step 1：写出带未知参数的函数 y = b + wx₁
│   ├── Step 2：定义损失 L(b,w) = (1/N)·Σ|ŷ - y|
│   ├── Step 3：梯度下降优化 θ ← θ - η·∇L
│   ├── 模型改进：从1天→7天→28天→56天的特征
│   └── 关键概念：batch、epoch、超参数
│
├── 1.2 线性模型
│   ├── 1.2.1 分段线性曲线 (Piecewise Linear Curve)
│   │   ├── 线性模型的局限：只能画直线
│   │   ├── 用Hard Sigmoid叠加逼近任意曲线
│   │   ├── 用Sigmoid函数逼近Hard Sigmoid
│   │   ├── 调整w/b/c制造不同形状的Sigmoid
│   │   └── 多个特征时的矩阵表示 r = b + Wx
│   ├── 1.2.2 模型变形
│   │   ├── ReLU = 两个Hard Sigmoid的组合
│   │   ├── Sigmoid vs ReLU的对比
│   │   ├── 激活函数的通用概念
│   │   └── 多层网络 = 深度学习的雏形
│   └── 1.2.3 机器学习框架
│       ├── 步骤1：写出带未知参数θ的函数 f_θ(x)
│       ├── 步骤2：定义损失函数 L(θ)
│       └── 步骤3：优化 θ* = argmin_θ L(θ)
```

> **本章与前后的联系：** 第1章是全书基础，建立"三步走"框架；第2章讲解训练不顺时的诊断方法；第3章深入优化和分类。

## 1.1 机器学习到底在做什么？

### 一句话定义

**机器学习 = 让机器自动找一个复杂的函数。**

人类能轻松做但写不出规则的事情，让机器从数据中学出一个"输入→输出"的映射函数。

### 三类机器学习任务

| 任务类型 | 输出形式 | 实例 |
|---------|---------|------|
| **回归(Regression)** | 一个数值(标量) | 预测明日PM2.5浓度、股价预测 |
| **分类(Classification)** | 从预设选项中选一个 | 垃圾邮件检测、AlphaGo(19×19选一) |
| **结构化学习(Structured)** | 有结构的对象 | 生成图片、写文章、翻译 |

### 为什么叫"学习"？

传统编程：人类写规则 → 计算机执行 → 得到结果
机器学习：人类给数据+答案 → 计算机自己找出规则 → 得到模型

> **核心类比：** 传统编程像老师一字一句教学生做题，机器学习像给一堆练习题和答案，学生自己总结解题方法。

## 1.2 案例学习：预测YouTube观看次数

### 背景设定

有一个YouTube频道，我们知道2017-2020年每天的观看次数。**目标：** 找一个函数，输入历史信息，输出明天的观看次数，帮他预测收益。

### 第一步：写出带未知参数的函数

机器学习的起点是一个**猜测**——你猜测输入和输出之间有什么关系：

$$y = b + wx_1$$

其中：
- $y$：要预测的值（明天观看次数）
- $x_1$：已知的特征（昨天观看次数）
- $w$：**权重(weight)**——未知参数，控制$x_1$的影响力
- $b$：**偏置(bias)**——未知参数，控制基准值

> 这个带未知参数的函数就叫**模型(model)**。$x_1$叫**特征(feature)**。

**为什么猜$y = b + wx_1$？** 这是**领域知识(domain knowledge)**——猜今天的观看次数大概和昨天差不多，乘个系数再微调。猜测不一定对，后面可以通过数据修正。

### 第二步：定义损失(Loss)

损失$L(b,w)$是一个函数，输入是$b$和$w$，输出是"这组参数有多差"。**损失越小 = 参数越好。**

#### 计算过程（以具体数值为例）

假设$b=500, w=1$，预测函数变成$y = 500 + x_1$。

1. 取2017年1月1日的观看次数$x_1=4800$
2. 预测值$\hat{y} = 500 + 1 \times 4800 = 5300$
3. 真实值（标签label）$y = 4900$
4. 误差$e_1 = |y - \hat{y}| = |4900 - 5300| = 400$

对所有训练数据（3年≈1095天）都算一遍误差，取平均：

$$L = \frac{1}{N}\sum_{n=1}^{N} e_n$$

#### 两种常用误差计算方式

| 名称 | 公式 | 特点 |
|------|------|------|
| **MAE**(Mean Absolute Error) | $e = \|\hat{y} - y\|$ | 对离群点不敏感 |
| **MSE**(Mean Squared Error) | $e = (\hat{y} - y)^2$ | 对大误差惩罚更重 |

> 当$y$和$\hat{y}$都是概率分布时，常用**交叉熵(cross-entropy)**作为损失。

### 第三步：优化——梯度下降(Gradient Descent)

把不同$w$和$b$组合的损失画出来，得到**误差表面(error surface)**——像一个山谷地形图。红色=损失大(差)，蓝色=损失小(好)。

我们的目标：找到$w^*$和$b^*$，使$L$最小。

#### 梯度下降的直观理解

想象你在浓雾中站在山坡上，想走到山谷最低点：
1. **环顾四周**（计算当前位置的梯度=最陡上坡方向）
2. **往最陡下坡方向走一步**（沿梯度反方向移动）
3. **重复**直到走不动

#### 单参数情况下的数学表述

假设只有$w$一个参数，$b$已知：

1. 随机选初始点$w_0$
2. 计算$\frac{\partial L}{\partial w}\big|_{w=w_0}$（$w_0$处的斜率）
   - 斜率<0（左高右低）→ 增大$w$可使损失变小
   - 斜率>0（左低右高）→ 减小$w$可使损失变小
3. 更新参数：

$$w_1 \leftarrow w_0 - \eta \cdot \frac{\partial L}{\partial w}\bigg|_{w=w_0}$$

其中$\eta$是**学习率(learning rate)**——自己设定的超参数，控制步伐大小。

> **超参数(hyperparameter)**：需要人手动设定、机器不会自己找的参数。

#### 梯度下降会卡在局部最小值吗？

如果走到一个地方梯度恰好为0（左右都比这里高），参数不再更新——这就是**局部最小值(local minimum)**。

但真正的**全局最小值(global minimum)**可能在别处。

> **重要预判（后面第3章会展开）：** 在高维空间中，真正的局部最小值其实很少见，梯度下降卡住的主要原因是**鞍点(saddle point)**。李宏毅老师明确说："局部最小值是一个假问题，做梯度下降时真正面对的难题不是局部最小值。"

#### 两个参数时的梯度下降

同时算$w$和$b$的偏导数，组成梯度向量，同时更新：

$$w_1 \leftarrow w_0 - \eta \frac{\partial L}{\partial w}, \quad b_1 \leftarrow b_0 - \eta \frac{\partial L}{\partial b}$$

在PyTorch等深度学习框架中，微分是**自动计算**的——你不需要手算导数！

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# ====== Step 1: 定义模型 ======
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 输入1个特征，输出1个值
    def forward(self, x):
        return self.linear(x)

model = LinearModel()
print(f"初始w={model.linear.weight.item():.4f}, b={model.linear.bias.item():.4f}")

# ====== Step 2: 定义损失 ======
criterion = nn.MSELoss()

# ====== Step 3: 定义优化器 ======
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# 造数据: y = 2x + 1 + 噪声
np.random.seed(42)
x = torch.linspace(-5, 5, 200).reshape(-1, 1)
y_true = 2 * x + 1 + torch.randn(200, 1) * 0.8

# 训练
losses = []
for epoch in range(150):
    y_pred = model(x)
    loss = criterion(y_pred, y_true)
    losses.append(loss.item())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 30 == 0:
        w, b = model.linear.weight.item(), model.linear.bias.item()
        print(f"Epoch {epoch+1:3d}: Loss={loss.item():.4f}, w={w:.4f}, b={b:.4f}")

print(f"\n真实参数: w=2.0, b=1.0")
print(f"学到参数: w={model.linear.weight.item():.4f}, b={model.linear.bias.item():.4f}")

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(losses); axes[0].set_title("Loss下降曲线"); axes[0].set_xlabel("Epoch")
axes[1].scatter(x, y_true, alpha=0.4, s=10)
axes[1].plot(x, model(x).detach(), "r", linewidth=2, label="拟合线")
axes[1].legend(); axes[1].set_title("数据与拟合结果")
# 画误差表面（等高线模拟）
ws = np.linspace(-1, 5, 50); bs = np.linspace(-3, 5, 50)
W, B = np.meshgrid(ws, bs)
Z = np.mean((W * x.numpy() + B - y_true.numpy())**2, axis=(0,1)).reshape(50,50)
axes[2].contourf(W, B, np.log(Z), levels=20, cmap="viridis")
axes[2].scatter([2], [1], c="r", s=100, marker="*", label="真实参数")
axes[2].scatter([model.linear.weight.item()], [model.linear.bias.item()], c="blue", s=80, marker="o", label="学到参数")
axes[2].legend(); axes[2].set_xlabel("w"); axes[2].set_ylabel("b"); axes[2].set_title("误差表面(log scale)")
plt.tight_layout(); plt.show()

### 模型改进：从1天到7天

最初的模型$y=b+wx_1$（只看昨天）在训练集上损失为480，测试集上为580。

观察数据发现：**观看人数有7天周期**（周五周六特别低）。于是改进模型：

$$y = b + \sum_{j=1}^{7} w_j x_j$$

训练损失：480 → **380**。测试损失：580 → **490**。

继续扩展：
- 考虑28天：训练损失330，测试损失460
- 考虑56天：训练损失320，测试损失460（不再改善，到达极限）

> **关键教训：** 这些模型的共同点是$y = b + \sum w_j x_j$——输入特征乘权重求和。这就是**线性模型(linear model)**。不管怎么调$w_j$，永远只能画直线。

## 1.3 分段线性曲线：突破线性的限制

### 线性模型的根本局限

线性模型画出来永远是一条直线（或多维空间中的超平面）。但现实数据可能：
- $x_1$很小时，$y$随$x_1$增大
- $x_1$超过某个阈值后，$y$反而随$x_1$减小

这种关系线性模型**永远无法表示**——这叫**模型偏差(model bias)**。

### 用简单函数拼出复杂函数

一个关键数学事实：**任何连续函数都可以用足够多的分段线性曲线来逼近。**

分段线性曲线又可以用"Hard Sigmoid"（硬S型函数）叠加而成：

```
Hard Sigmoid = 先水平 → 再斜坡 → 再水平
红色复杂曲线 = 常数项 + HardSigmoid₁ + HardSigmoid₂ + HardSigmoid₃ + ...
```

每个Hard Sigmoid的**斜坡起点、坡度、高度**都不同，叠加起来就能逼近任何形状。

> **直觉类比：** 就像用乐高积木拼出复杂形状——每个积木(分段线性函数)很简单，但足够多就能拼出任何东西。

### 用Sigmoid函数逼近Hard Sigmoid

直接用分段函数不好求导。于是用**Sigmoid函数**（S型函数）来平滑近似：

$$y = c \cdot \sigma(b + wx_1) = c \cdot \frac{1}{1 + e^{-(b + wx_1)}}$$

**三个参数各控制什么？**
- **$w$（权重）**：改变斜率/坡度。$|w|$越大，S型越陡峭
- **$b$（偏置）**：左右平移S型曲线。$b$越大，曲线越往左移
- **$c$（缩放）**：改变S型曲线的高度

不同$(w,b,c)$组合 → 不同形状的Sigmoid → 叠加 → 逼近任意连续函数！

$$y = b + \sum_i c_i \cdot \sigma(b_i + w_i x_1)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x, w, b, c):
    return c / (1 + np.exp(-(b + w * x)))

x = np.linspace(-10, 10, 500)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 改变w：影响坡度
for w in [0.5, 1, 2, 5]:
    axes[0].plot(x, sigmoid(x, w, 0, 1), label=f"w={w}")
axes[0].set_title("改变w（影响坡度/斜率）")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 改变b：影响左右位置
for b in [-4, -1, 1, 4]:
    axes[1].plot(x, sigmoid(x, 1, b, 1), label=f"b={b}")
axes[1].set_title("改变b（影响左右位置）")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# 改变c：影响高度
for c in [0.5, 1, 2, 4]:
    axes[2].plot(x, sigmoid(x, 1, 0, c), label=f"c={c}")
axes[2].set_title("改变c（影响高度）")
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle("Sigmoid函数: y = c/(1+e^{-(b+wx)})", fontsize=14)
plt.tight_layout(); plt.show()

### 多个特征时的矩阵表示

不只用一个$x_1$，可以用前3天的数据$x_1,x_2,x_3$，每个Sigmoid函数都综合考虑所有特征：

第$i$个Sigmoid的输入：$r_i = b_i + w_{i1}x_1 + w_{i2}x_2 + w_{i3}x_3$

用矩阵一次性写出所有$i$的$r$：

$$\mathbf{r} = \mathbf{b} + \mathbf{W}\mathbf{x}$$

$$\begin{bmatrix} r_1 \\ r_2 \\ r_3 \end{bmatrix} = \begin{bmatrix} b_1 \\ b_2 \\ b_3 \end{bmatrix} + \begin{bmatrix} w_{11} & w_{12} & w_{13} \\ w_{21} & w_{22} & w_{23} \\ w_{31} & w_{32} & w_{33} \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}$$

然后对所有$r_i$过Sigmoid得到$a_i$：$\mathbf{a} = \sigma(\mathbf{r})$

最终输出：$y = b + \mathbf{c}^T\mathbf{a} = b + \sum_i c_i a_i$

> **把所有参数"拼"成一个向量$\theta$：** $\theta = [w_{11}, w_{12}, ..., b_1, b_2, ..., c_1, c_2, ..., b]$。这样损失函数就写成$L(\theta)$，梯度下降更新$\theta \leftarrow \theta - \eta \nabla L$。

### 批量训练：一次算多少数据？

实际训练不会一次用全部$N$条数据算梯度（计算太慢），而是：

1. 把$N$条数据随机分成多个**批量(batch)**，每个batch有$B$条
2. 每次只用一个batch的数据算损失$L_1$和梯度
3. 用这个梯度更新参数
4. 换下一个batch，重复

| 概念 | 含义 | 例子 |
|------|------|------|
| **Batch Size ($B$)** | 每个batch的样本数 | 设$B=10$ |
| **一次更新(Update)** | 用某个batch更新一次参数 | — |
| **一个回合(Epoch)** | 把所有batch都看过一遍 | $N=10000, B=10$ → 1 epoch = 1000次更新 |

> **关键区分：** Epoch和Update是不同的！一个Epoch中更新的次数 = N/B。所以不能说"训练了100个epoch"，还要知道batch size是多少。

## 1.4 模型变形：从Sigmoid到ReLU

Hard Sigmoid其实可以用**两个ReLU的叠加**来等价表示：

$$\text{ReLU}(x) = c \cdot \max(0, b + wx)$$

- 如果$b+wx < 0$：输出0（水平段）
- 如果$b+wx > 0$：输出$b+wx$（斜坡段）

2个ReLU叠在一起 = 1个Hard Sigmoid。所以用ReLU也完全可以实现分段线性逼近！

| 激活函数 | 公式 | 特点 |
|---------|------|------|
| Sigmoid | $\frac{1}{1+e^{-x}}$ | 输出(0,1)，平滑，但梯度可能消失 |
| ReLU | $\max(0, x)$ | 计算快，缓解梯度消失，最常用 |
| Hard Sigmoid | 分段函数 | 需要2个ReLU合成 |

**实验结果：** 100个ReLU比10个ReLU好很多，但1000个相比100个在测试集上不再提升——模型能力有上限。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 理解ReLU
x = torch.linspace(-5, 5, 200)
relu = nn.ReLU()
sigmoid = nn.Sigmoid()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, relu(x), linewidth=2, label="ReLU: max(0,x)")
axes[0].plot(x, sigmoid(x), linewidth=2, label="Sigmoid: 1/(1+e^{-x})")
axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_title("ReLU vs Sigmoid")

# 演示"多层"的概念
class OneLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 100), nn.ReLU(), nn.Linear(100, 1))
    def forward(self, x): return self.net(x)

class ThreeLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 100), nn.ReLU(),
            nn.Linear(100, 100), nn.ReLU(),
            nn.Linear(100, 100), nn.ReLU(),
            nn.Linear(100, 1)
        )
    def forward(self, x): return self.net(x)

# 用它们拟合一个复杂的非线性函数
x_train = torch.linspace(-3, 3, 150).reshape(-1, 1)
y_train = torch.sin(2*x_train) + 0.5*torch.cos(5*x_train) + 0.2*torch.randn(150, 1)

def train_model(model, name, epochs=500):
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(x_train), y_train)
        loss.backward()
        opt.step()
    return model

model1 = train_model(OneLayer(), "1层")
model3 = train_model(ThreeLayer(), "3层")

# 画图
x_test = torch.linspace(-4, 4, 300).reshape(-1, 1)
axes[1].scatter(x_train, y_train, s=5, alpha=0.4, label="训练数据")
axes[1].plot(x_test, model1(x_test).detach(), label="1层网络(100个ReLU)", linewidth=2)
axes[1].plot(x_test, model3(x_test).detach(), label="3层网络(300个ReLU)", linewidth=2)
axes[1].legend(); axes[1].set_title("深度 = 更多层，能拟合更复杂的函数")
plt.tight_layout(); plt.show()

print("深度学习的'深度'就是指层数。")
print("每多一层，就是对数据多做一次非线性变换。")
print("层数越多，函数的灵活性(flexibility)越强。")

## 1.5 机器学习三步骤框架（全书统一定式）

这一框架贯穿全书20章，理解它才能理解后续所有内容：

| 步骤 | 干什么 | 输出 | 关键问题 |
|------|--------|------|----------|
| **Step 1** | 写出带未知参数$\theta$的函数$f_\theta(x)$ | 模型架构 | 这个函数池够大吗？能包含真实规律吗？ |
| **Step 2** | 定义损失$L(\theta)$衡量参数好坏 | 损失函数 | 损失函数的选择是否匹配任务？ |
| **Step 3** | 用梯度下降找$\theta^* = \arg\min L(\theta)$ | 最优参数 | 梯度下降真的找到了最优解吗？ |

后续每一章都可以套用这个框架：
- CNN（第4章）：Step 1换成卷积网络 → Step 2/3不变
- GAN（第8章）：Step 1=生成器+辨别器 → Step 2=对抗损失 → Step 3=交替优化
- 强化学习（第14章）：Step 1=策略网络 → Step 2=负期望奖励 → Step 3=Policy Gradient

## 常见误区与注意事项

1. ❌ **"机器学习就是调包跑模型"** → 调包之前要先理解三步框架，知道每一步在做什么
2. ❌ **"模型越复杂越好"** → 数据少时复杂模型会过拟合（见第2章）
3. ❌ **"学习率越大训练越快"** → 学习率太大直接不收敛，太小训练太慢
4. ❌ **"梯度下降一定会找到最优解"** → 可能卡在局部最小值或鞍点（见第3章）
5. ❌ **"损失函数只能是MSE"** → 分类任务需要用交叉熵（见第3章3.6节）
6. ❌ **"Sigmoid和ReLU没区别"** → ReLU计算更快、缓解梯度消失，是现代网络的默认选择
7. ❌ **"batch size无所谓"** → batch size影响训练速度、稳定性和泛化能力（见第3章3.2节）
8. ❌ **"机器学习可以没有任何领域知识"** → 至少要知道输入和输出分别是什么，特征怎么选

## 本章核心收获

1. **机器学习 = 找函数**：回归输出数值，分类输出类别，结构化学习输出复杂对象
2. **三步框架**：定义模型 → 定义损失 → 梯度下降优化
3. **线性到非线性**：多个Sigmoid/ReLU叠加可以逼近任意连续函数
4. **矩阵化**：所有参数拼成向量$\theta$，用矩阵乘法高效计算
5. **Batch vs Epoch**：Batch是每次更新用的数据量，Epoch是看完所有数据一遍
6. **深度学习的"深度"**：多层线性变换+激活函数的嵌套
7. 基本概念：特征、标签、模型、损失、梯度、学习率、超参数、激活函数

> 📖 **下一章预告：** 训练不顺利怎么办？第2章教你系统性地诊断和解决问题。